In [21]:
import json
from openai import OpenAI
from IPython.display import display, Markdown

In [22]:
def get_shipping_cost(destination_zone: str, weight: float) -> dict:
    rates = {
        "zone_1": 5.0,
        "zone_2": 10.0,
        "zone_3": 15.0
    }
    base_rate = rates.get(destination_zone.lower(), 10)
    total_cost = round(base_rate + (weight * 1.25), 2)
    return {
        "destination_zone": destination_zone,
        "weight": weight,
        "total_cost": total_cost    
    }

In [24]:
display(Markdown("### Shipping Cost Example"))
example_cost = get_shipping_cost("zone_2", 5.5)
display(Markdown(f"Destination Zone: {example_cost['destination_zone']}"))
display(Markdown(f"Weight: {example_cost['weight']}"))
display(Markdown(f"Total Cost: ${example_cost['total_cost']}"))

### Shipping Cost Example

Destination Zone: zone_2

Weight: 5.5

Total Cost: $16.88

In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_shipping_cost",
            "description": "Calculate the shipping cost based on the destination zone and weight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_zone": {
                        "type": "string",
                        "description": "The destination zone for the shipment."
                    },
                    "weight": {
                        "type": "number",
                        "description": "The weight of the shipment."
                    }
                },
                "required": ["destination_zone", "weight"]
            }
        }
    }
]

In [20]:
client = OpenAI()

In [26]:
messages = [
    {
        "role": "user",
        "content": "How much is the shipping cost for zone_2 with a weight of 4.2 pounds"
    }
]

In [31]:
response = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=messages,
    tools=tools
)
assistant_response = response.choices[0].message
messages.append(assistant_response)

In [38]:
if assistant_response.tool_calls:
    for tool_call in assistant_response.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        
        if fn_name == "get_shipping_cost":
            result = get_shipping_cost(
                destination_zone=fn_args["destination_zone"],
                weight=fn_args["weight"]
            )
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
    
    final_response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=messages,
        tools=tools
    )
    print(display(Markdown(final_response.choices[0].message.content)))

else:
    print(display(Markdown(assistant_response.content)))

The shipping cost for **zone_2** with a weight of **4.2 lb** is **$15.25**.

None
